# Chapter 3 — Embedding Models

> **TL;DR** — The embedder decides what's findable. This chapter implements: contrastive training with **in-batch + hard negatives** (the real quality lever), **Matryoshka** truncation and adaptive two-pass retrieval, **int8/binary quantization** with measured recall loss, and a hard-negative **mining loop**. All runnable with `sentence-transformers` + NumPy.

`pip install sentence-transformers datasets numpy scikit-learn`

---

## 3.1 What an embedding actually is (and why dense misses exact tokens)

A dense encoder is a transformer whose token outputs are **pooled** (mean or CLS) into one vector. Mean pooling literally averages token vectors:

In [1]:
# This function takes the raw, token-by-token output of a Transformer model 
# and squashes it into a single "Sentence Embedding."

import torch
def mean_pool(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()            # (B, T, 1)
    summed = (token_embeddings * mask).sum(1)              # ignore padding
    counts = mask.sum(1).clamp(min=1e-9)
    return summed / counts 

#### The Padding Problem

When we send text through AI models, we usually send it in "batches" (e.g., 32 sentences at a time). Because neural networks require uniform matrix sizes, all 32 sentences must be exactly the same length.
We achieve this by adding blank "Padding Tokens" to the end of the shorter sentences.

If we simply averaged all the word vectors together using a standard average function, the fake padding tokens would drag down the math and ruin the meaning of the sentence. This code calculates an average while strictly ignoring the padding.

In [4]:
from sentence_transformers import SentenceTransformer, util

m = SentenceTransformer("all-MiniLM-L6-v2")
docs = ["The refund window is 30 days.", "Error code X7-4420B indicates a sensor fault."]

for q in ["how long to get money back", "x7-4420B"]:
    sims = util.cos_sim(m.encode(q), m.encode(docs))[0]
    print(q, "->", [f"{s:.3f}" for s in sims])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1655.19it/s]


how long to get money back -> ['0.584', '-0.045']
x7-4420B -> ['-0.030', '0.511']


**This averaging is why dense retrieval blurs rare exact tokens.** A part number `X7-4420B` is one rare token; averaged with 399 others, its contribution to the final vector is ~1/400 and semantically it had no strong learned representation to begin with. The signal is drowned. That's the mathematical reason you *must* add BM25/sparse for exact-match queries — no embedding tweak fixes averaging.

## 3.3 Contrastive loss + in-batch negatives (the workhorse)

When we want to train an AI to understand which sentences mean the same thing, we have to teach it what "similar" looks like, but just as importantly, we have to teach it what "different" looks like.

This process is called **Contrastive Learning**. We are teaching the AI by contrasting good matches against bad matches.

In [5]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("all-MiniLM-L6-v2")

# each example is (anchor, positive). Batch supplies the negatives implicitly.
train = [InputExample(texts=[q, p]) for q, p in [
    ("how long for a refund", "Refunds are processed within 30 days of return."),
    ("who leads the company",  "Jane Doe has served as CEO since 2019."),
    # ... thousands more
]]
loader = DataLoader(train, shuffle=True, batch_size=32)      # bigger batch = more negatives = better
loss = losses.MultipleNegativesRankingLoss(model)           # InfoNCE with in-batch negatives
model.fit(train_objectives=[(loader, loss)], epochs=1, warmup_steps=100)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2797.92it/s]
e:\Learning\Preparation\Retrieval-Augmented-Generation\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


The dominant training objective is **Multiple Negatives Ranking Loss (MNRL / InfoNCE)**: in a batch of `(query_i, positive_i)` pairs, treat every *other* positive in the batch as a negative for query_i. Softmax cross-entropy over similarities pulls each query to its positive and pushes it from the batch's other passages — for free.

The InfoNCE loss it computes (τ = temperature/scale):

```
L = -log[ exp(sim(q, p⁺)/τ) / Σ_j exp(sim(q, p_j)/τ) ]      # p_j = all passages in batch
```

The formula simply tells the model: "Adjust your weights so that the correct answer gets as close to 100% of the total points as possible, leaving 0% for the 31 other answers."

**Batch size is a hyperparameter that changes quality**, not just speed: a batch of 64 gives 63 negatives per query; a batch of 8 gives 7. More (harder-to-distinguish) negatives → sharper embeddings. This is why embedding training uses large batches / gradient accumulation.

## 3.4 Hard-negative mining — where recall actually moves

In-batch negatives are mostly *easy* (random unrelated passages). The model learns the fine distinctions only from **hard negatives**: passages that look relevant but aren't. You mine them with the *current* model, then retrain.

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.util import cos_sim

def mine_hard_negatives(model, queries, positives, corpus, corpus_ids,
                        top_k=50, n_neg=4, margin=0.05):
    """For each query, retrieve top_k, take high-ranked NON-positives as hard negs.
       Filter false negatives: skip candidates ~as similar as the positive."""
    C = model.encode(corpus, normalize_embeddings=True, convert_to_numpy=True)
    triplets = []
    for q, pos in zip(queries, positives):
        qv = model.encode(q, normalize_embeddings=True)
        pos_sim = float(cos_sim(qv, model.encode(pos))[0][0])
        sims = C @ qv
        order = np.argsort(-sims)[:top_k]
        negs = []
        for idx in order:
            cand = corpus[idx]
            if cand == pos:                       continue   # it's the positive
            if sims[idx] > pos_sim - margin:      continue   # likely FALSE negative → skip
            negs.append(cand)
            if len(negs) >= n_neg: break
        for n in negs:
            triplets.append(InputExample(texts=[q, pos, n]))  # (anchor, pos, hard-neg)
    return triplets

In [8]:
queries    = ["how long for a refund", "who leads the company"]              # List[str], len Q
positives  = ["Refunds are processed within 30 days of return.",              # List[str], len Q
              "Jane Doe has served as CEO since 2019."]                      # positives[i] answers queries[i]
corpus     = [positives[0], positives[1], "Our offices are closed on public holidays."]  # List[str], len N — every candidate passage, positives included
corpus_ids = [f"doc_{i}" for i in range(len(corpus))]                        # List[str], len N — external id per corpus entry

In [9]:
# retrain with explicit hard negatives (MNRL also accepts triplets: [anchor, pos, neg])
triplets = mine_hard_negatives(model, queries, positives, corpus, corpus_ids)
loader = DataLoader(triplets, shuffle=True, batch_size=16)
model.fit([(loader, losses.MultipleNegativesRankingLoss(model))], epochs=1)

Step,Training Loss


**The two things this code gets right that people miss:**
1. **Mining uses the current model** — hard negatives are defined *relative to what the model currently confuses*. Iterate: train → re-mine → train (2–3 rounds).
2. **False-negative filtering** (`sims[idx] > pos_sim - margin`) — a "hard negative" that's actually relevant but unlabeled will teach the model that a correct answer is wrong. Skipping near-positive candidates is essential and the detail that separates someone who *read* about hard negatives from someone who *did* it.

> "The loss barely matters versus the negatives. In-batch negatives are easy; we mine hard negatives with the current model and filter false negatives by a margin below the positive's similarity. Two or three mine-retrain rounds is where recall moves."

## 3.5 Matryoshka: one model, many dimensions

In [13]:
print(model.get_embedding_dimension())

384


In [ ]:

# training nested loss
from sentence_transformers import losses

base = losses.MultipleNegativesRankingLoss(model)

mrl = losses.MatryoshkaLoss(
    model,
    base,
    matryoshka_dims=[384, 256, 128, 64]
)

model.fit([(loader, mrl)], epochs=1)

Step,Training Loss


In [14]:
# Using it — truncate + renormalize

def truncate(vecs, dim):
    v = vecs[:, :dim]
    return v / np.linalg.norm(v, axis=1, keepdims=True)   # MUST renormalize after slicing


In [15]:
# Adaptive two-pass retrieval (the payoff)

def adaptive_search(query, full_matrix, coarse_dim=128, shortlist=200, k=10):
    qv_full = model.encode(query, normalize_embeddings=True)
    # pass 1: cheap coarse search over truncated vectors across ALL N
    coarse_docs = truncate(full_matrix, coarse_dim)
    qv_coarse   = truncate(qv_full[None, :], coarse_dim)[0]
    cand = np.argsort(-(coarse_docs @ qv_coarse))[:shortlist]
    # pass 2: precise rescore of the shortlist with FULL dims
    fine = np.argsort(-(full_matrix[cand] @ qv_full))[:k]
    return cand[fine]

## 3.6 Quantization with measured recall loss

In [16]:
import numpy as np

def quantize_int8(vecs):
    lo, hi = vecs.min(0), vecs.max(0)                      # per-dimension range
    scale = (hi - lo) / 255.0
    q = np.round((vecs - lo) / scale).astype(np.uint8)
    return q, lo, scale
def dequantize_int8(q, lo, scale): return q.astype(np.float32) * scale + lo

def quantize_binary(vecs):  return np.packbits(vecs > 0, axis=1)   # 1 bit/dim → uint8 packed
def hamming(a, b):          return np.unpackbits(a ^ b, axis=1).sum(1)

def recall_at_k(true_ids, got_ids, k):
    return len(set(true_ids[:k]) & set(got_ids[:k])) / k

# measure the tradeoff on your own vectors
def eval_quant(matrix, queries_v, k=10, shortlist=100):
    fp32_top = [np.argsort(-(matrix @ q))[:k] for q in queries_v]
    # binary + rescore
    B, qB = quantize_binary(matrix), quantize_binary(queries_v)
    rec = []
    for i, q in enumerate(queries_v):
        cand = np.argsort(hamming(qB[i][None], B))[:shortlist]     # wide, cheap Hamming pass
        fine = cand[np.argsort(-(matrix[cand] @ q))[:k]]           # rescore with fp32
        rec.append(recall_at_k(list(fp32_top[i]), list(fine), k))
    print(f"binary+rescore recall@{k}: {np.mean(rec):.3f}  |  mem: 32x smaller")
